# Lantern Second Pass on Local ANTARES Data — v8

This version follows `investigating_second_pass.ipynb`, with one update to the target definition:

> **Eligible ANTARES loci separated by less than 3 arcsec are linked with friends-of-friends (FoF), and each connected component is treated as one Second-Pass target.**

For an isolated locus, one locus is still one target, exactly as in v7.

Workflow:

1. read cleaned loci and alerts;
2. keep alerts from **2026-05-27 onward**;
3. require `lsst_diaSource_band`;
4. keep First-Cut loci with `max_score > 0.9691`;
5. exclude the same locus-level counter anomalies used in v7;
6. run **3 arcsec FoF on eligible locus coordinates**;
7. combine all alerts from all member loci of each target;
8. build the same Second-Pass features at the target level;
9. apply `second_pass_first_test.pkl`;
10. keep candidates with `second_pass_score > 0.8`.

For multi-locus targets, the original ANTARES `locus_id`s are preserved in `member_locus_ids`.

In [ ]:
import os
import ast
import json
import pickle
from collections.abc import Mapping

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord, search_around_sky
import astropy.units as u

# -------------------- INPUTS --------------------

LOCI_FILE = "antares_data_clean_from_20260527/loci/loci_00000.parquet"
ALERTS_FILE = "antares_data_clean_from_20260527/alerts/alerts_00000.parquet"
MODEL_FILE = "second_pass_first_test.pkl"

# First-Cut consistency check used by investigating_second_pass.ipynb
FIRST_CUT_THRESHOLD = 0.9691

# Include alerts dated May 27, 2026 or later
START_MJD = 61187.0

# v8 target definition: FoF linking length between locus coordinates
LOCUS_LINKING_LENGTH = 3.0  # arcsec

# Run on 10% of eligible targets.
# Set to 1.0 (or None) to run all eligible targets.
SAMPLE_FRAC = 1.0 #0.10
RANDOM_STATE = 42

# Investigation notebook's working Second-Pass cut
SECOND_PASS_THRESHOLD = 0.80

OUTPUT_DIR = "second_pass_output_v8"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:

# Load data and model

loci = pd.read_parquet(LOCI_FILE)
alerts = pd.read_parquet(ALERTS_FILE)

with open(MODEL_FILE, "rb") as f:
    model = pickle.load(f)

required_locus = {
    "locus_id", "ra", "dec",
    "max_score", "num_tagged_alerts",
}
required_alert = {
    "locus_id", "alert_id", "mjd", "alert_properties",
}

missing = required_locus - set(loci.columns)
if missing:
    raise KeyError(f"Missing locus columns: {sorted(missing)}")

missing = required_alert - set(alerts.columns)
if missing:
    raise KeyError(f"Missing alert columns: {sorted(missing)}")

if loci["locus_id"].duplicated().any():
    raise ValueError("LOCI_FILE contains duplicate locus_id rows.")

if alerts["alert_id"].duplicated().any():
    raise ValueError("ALERTS_FILE contains duplicate alert_id rows.")

print(f"Loci:   {len(loci):,}")
print(f"Alerts: {len(alerts):,}")
print(f"Model:  {MODEL_FILE}")


In [ ]:

# Helpers

def as_dict(value):
    if isinstance(value, Mapping):
        return value

    if isinstance(value, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(value)
                if isinstance(parsed, Mapping):
                    return parsed
            except Exception:
                pass

    return {}


def get_prop(value, key, default=np.nan):
    value = as_dict(value)
    result = value.get(key, default)
    return default if result is None else result


def has_prop(value, key):
    return key in as_dict(value)


def normalize_band(series):
    # The investigation notebook defines this normalization but does not
    # explicitly call it. We apply it here so FilterLabel(...) and plain
    # "g"/"r"/... values are handled consistently.
    s = series.astype(str)

    extracted = s.str.extract(
        r"band=['\"]([ugrizy])['\"]",
        expand=False,
    )
    plain = s.str.extract(
        r"^([ugrizy])$",
        expand=False,
    )

    return extracted.fillna(plain)


def spherical_mean_radec(ra_deg, dec_deg):
    """Mean sky position, robust to RA wrap-around."""
    ra = np.deg2rad(np.asarray(ra_deg, dtype=float))
    dec = np.deg2rad(np.asarray(dec_deg, dtype=float))

    x = np.cos(dec) * np.cos(ra)
    y = np.cos(dec) * np.sin(ra)
    z = np.sin(dec)

    x = np.nanmean(x)
    y = np.nanmean(y)
    z = np.nanmean(z)

    ra_mean = np.arctan2(y, x)
    dec_mean = np.arctan2(z, np.sqrt(x**2 + y**2))

    return np.rad2deg(ra_mean) % 360.0, np.rad2deg(dec_mean)


## 1. Restrict the alert sample

In [ ]:

alerts = alerts.copy()
alerts["mjd"] = pd.to_numeric(alerts["mjd"], errors="coerce")

n_initial = len(alerts)

# Analysis is restricted to the confirmed Lantern operating period.
alerts = alerts[alerts["mjd"] >= START_MJD].copy()
n_after_date = len(alerts)

# Match investigating_second_pass.ipynb:
# only alerts carrying an LSST band enter the Second-Pass analysis.
has_band = alerts["alert_properties"].map(
    lambda p: has_prop(p, "lsst_diaSource_band")
)
alerts = alerts[has_band].copy()

print(f"Initial alerts:                    {n_initial:,}")
print(f"Alerts on/after MJD {START_MJD}:       {n_after_date:,}")
print(f"Alerts with lsst_diaSource_band:  {len(alerts):,}")

if len(alerts):
    print(f"MJD range used: {alerts['mjd'].min():.6f} - {alerts['mjd'].max():.6f}")


## 2. Select First-Cut loci and flag counter anomalies

In [ ]:

# The local 10k sample was already downloaded from the Lantern-tagged
# population, so no additional local tag check is needed.

first_cut = loci.copy()

first_cut["first_cut_max_score"] = pd.to_numeric(
    first_cut["max_score"],
    errors="coerce",
)
first_cut["n_tagged"] = pd.to_numeric(
    first_cut["num_tagged_alerts"],
    errors="coerce",
)

first_cut = first_cut[
    first_cut["first_cut_max_score"] > FIRST_CUT_THRESHOLD
].copy()

print(f"First-Cut loci: {len(first_cut):,}")


In [ ]:

# n_lsst_alerts = number of retained LSST alert rows for each locus.
# This is equivalent to the investigation notebook's local
# alert_id = 0,1,2,... followed by max(alert_id) + 1.

alert_counts = (
    alerts[alerts["locus_id"].isin(first_cut["locus_id"])]
    .groupby("locus_id")
    .size()
    .rename("n_lsst_alerts")
)

first_cut["n_lsst_alerts"] = (
    first_cut["locus_id"]
    .map(alert_counts)
    .fillna(0)
    .astype(int)
)

first_cut["missing_counter_flag"] = first_cut["n_tagged"].isna()
first_cut["bad_counter_flag"] = (
    first_cut["n_tagged"] > first_cut["n_lsst_alerts"]
)
first_cut["no_alert_flag"] = (
    first_cut["n_lsst_alerts"] == 0
)

flagged = first_cut[
    first_cut["missing_counter_flag"]
    | first_cut["bad_counter_flag"]
    | first_cut["no_alert_flag"]
].copy()

print(f"Missing n_tagged:        {first_cut['missing_counter_flag'].sum():,}")
print(f"n_tagged > n_lsst:       {first_cut['bad_counter_flag'].sum():,}")
print(f"No retained LSST alert:  {first_cut['no_alert_flag'].sum():,}")

if len(flagged):
    display(
        flagged[
            [
                "locus_id", "ra", "dec",
                "n_tagged", "n_lsst_alerts",
                "missing_counter_flag",
                "bad_counter_flag",
                "no_alert_flag",
            ]
        ]
    )

flagged.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_flagged_loci.csv",
    ),
    index=False,
)

eligible = first_cut[
    ~first_cut["missing_counter_flag"]
    & ~first_cut["bad_counter_flag"]
    & ~first_cut["no_alert_flag"]
].copy()

print(f"Eligible First-Cut loci: {len(eligible):,}")


## 3. Group eligible loci into targets with 3″ FoF

In [ ]:
# A target is a connected component of eligible loci using a 3" linking length.
# Example: A-B < 3" and B-C < 3" -> A, B, C are one target,
# even if A-C > 3".

eligible = eligible.reset_index(drop=True).copy()

coord_good = (
    pd.to_numeric(eligible["ra"], errors="coerce").notna()
    & pd.to_numeric(eligible["dec"], errors="coerce").notna()
)

if (~coord_good).any():
    raise ValueError(
        f"{(~coord_good).sum()} eligible loci have invalid RA/Dec."
    )

coords = SkyCoord(
    ra=eligible["ra"].to_numpy() * u.deg,
    dec=eligible["dec"].to_numpy() * u.deg,
)

idx1, idx2, sep2d, _ = search_around_sky(
    coords,
    coords,
    LOCUS_LINKING_LENGTH * u.arcsec,
)

# Remove self-matches and symmetric duplicates.
pair_keep = idx1 < idx2
idx1 = idx1[pair_keep]
idx2 = idx2[pair_keep]
sep_arcsec = sep2d[pair_keep].arcsec

close_pairs = pd.DataFrame({
    "locus_id_1": eligible.iloc[idx1]["locus_id"].to_numpy(),
    "locus_id_2": eligible.iloc[idx2]["locus_id"].to_numpy(),
    "separation_arcsec": sep_arcsec,
}).sort_values("separation_arcsec").reset_index(drop=True)

print(
    f"Unique eligible-locus pairs within {LOCUS_LINKING_LENGTH:.1f} arcsec: "
    f"{len(close_pairs):,}"
)

# FoF / connected components.
parent = np.arange(len(eligible))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    ri = find(i)
    rj = find(j)
    if ri != rj:
        parent[rj] = ri


for i, j in zip(idx1, idx2):
    union(int(i), int(j))

roots = np.array([find(i) for i in range(len(eligible))])
components = {}
for row_index, root in enumerate(roots):
    components.setdefault(root, []).append(row_index)

# Stable target numbering.
ordered_components = sorted(
    components.values(),
    key=lambda members: min(
        eligible.iloc[members]["locus_id"].astype(str)
    ),
)

membership_rows = []
target_rows = []

for number, members in enumerate(ordered_components, start=1):
    target_id = f"target_{number:06d}"
    group = eligible.iloc[members].copy()

    member_ids = sorted(group["locus_id"].astype(str).tolist())
    target_ra, target_dec = spherical_mean_radec(
        group["ra"],
        group["dec"],
    )

    for locus_id in member_ids:
        membership_rows.append({
            "target_id": target_id,
            "locus_id": locus_id,
        })

    target_rows.append({
        "target_id": target_id,
        "ra": target_ra,
        "dec": target_dec,
        "n_loci": len(group),
        "member_locus_ids": member_ids,
        "first_cut_max_score": group["first_cut_max_score"].max(),
        "n_tagged": group["n_tagged"].sum(),
        "n_lsst_alerts_from_loci": group["n_lsst_alerts"].sum(),
    })

membership = pd.DataFrame(membership_rows)
targets_all = pd.DataFrame(target_rows)

print(f"Eligible loci:             {len(eligible):,}")
print(f"Targets after 3'' FoF:     {len(targets_all):,}")
print(f"Multi-locus targets:       {(targets_all['n_loci'] > 1).sum():,}")
print(
    "Loci in multi-locus targets: "
    f"{targets_all.loc[targets_all['n_loci'] > 1, 'n_loci'].sum():,}"
)

close_pairs.to_csv(
    os.path.join(OUTPUT_DIR, "close_locus_pairs_3arcsec.csv"),
    index=False,
)

membership.to_csv(
    os.path.join(OUTPUT_DIR, "target_membership.csv"),
    index=False,
)

targets_all.to_parquet(
    os.path.join(OUTPUT_DIR, "all_targets.parquet"),
    index=False,
)

### Multi-locus targets

In [ ]:
multi_targets = (
    targets_all[targets_all["n_loci"] > 1]
    .sort_values(
        ["n_loci", "target_id"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

multi_targets

## 4. Sample targets

In [ ]:
# Sample targets AFTER FoF grouping.

if SAMPLE_FRAC is None or SAMPLE_FRAC >= 1:
    targets = targets_all.copy()
else:
    n_sample = max(
        1,
        int(round(len(targets_all) * SAMPLE_FRAC)),
    )
    targets = targets_all.sample(
        n=n_sample,
        random_state=RANDOM_STATE,
    ).copy()

target_ids = set(targets["target_id"])

sample_membership = membership[
    membership["target_id"].isin(target_ids)
].copy()

# Each alert keeps its original locus_id and gains target_id.
target_alerts = alerts.merge(
    sample_membership,
    on="locus_id",
    how="inner",
)

print(f"Targets used:             {len(targets):,}")
print(f"Multi-locus targets used: {(targets['n_loci'] > 1).sum():,}")
print(f"Alerts used:              {len(target_alerts):,}")

# Sanity check: summed locus-level alert counts should equal direct target counts.
direct_counts = (
    target_alerts.groupby("target_id")
    .size()
    .rename("n_lsst_alerts_direct")
)

count_check = targets[
    ["target_id", "n_lsst_alerts_from_loci"]
].copy()

count_check["n_lsst_alerts_direct"] = (
    count_check["target_id"]
    .map(direct_counts)
    .fillna(0)
    .astype(int)
)

count_mismatch = (
    count_check["n_lsst_alerts_from_loci"]
    != count_check["n_lsst_alerts_direct"]
)

print(
    "Target alert-count mismatches: "
    f"{count_mismatch.sum():,}"
)

if count_mismatch.any():
    display(count_check[count_mismatch])

## 5. Build alert-level quantities

In [ ]:

def build_alert_features(alert_subset):
    p = alert_subset["alert_properties"]

    keys = {
        "band": "lsst_diaSource_band",
        "x": "lsst_diaSource_x",
        "y": "lsst_diaSource_y",
        "xErr": "lsst_diaSource_xErr",
        "yErr": "lsst_diaSource_yErr",
        "ra": "lsst_diaSource_ra",
        "dec": "lsst_diaSource_dec",
        "scienceFlux": "lsst_diaSource_scienceFlux",
        "psfFlux": "lsst_diaSource_psfFlux",
        "apFlux": "lsst_diaSource_apFlux",
        "extendedness": "lsst_diaSource_extendedness",
        "dipoleLength": "lsst_diaSource_dipoleLength",
        "ixx": "lsst_diaSource_ixx",
        "iyy": "lsst_diaSource_iyy",
        "ixxPSF": "lsst_diaSource_ixxPSF",
        "iyyPSF": "lsst_diaSource_iyyPSF",
    }

    out = alert_subset[
        ["target_id", "locus_id", "alert_id", "mjd"]
    ].copy()

    for name, key in keys.items():
        out[name] = p.map(
            lambda value, k=key: get_prop(value, k)
        )

    out["band"] = normalize_band(out["band"])

    numeric = [name for name in keys if name != "band"]
    out[numeric] = out[numeric].apply(
        pd.to_numeric,
        errors="coerce",
    )

    psf_trace = out["ixxPSF"] + out["iyyPSF"]
    src_trace = out["ixx"] + out["iyy"]

    out["moment_ext"] = np.where(
        psf_trace != 0,
        src_trace / psf_trace,
        np.nan,
    )

    out["template_flux"] = (
        out["scienceFlux"] - out["psfFlux"]
    )

    out["flux_ext"] = np.where(
        out["psfFlux"] != 0,
        out["apFlux"] / out["psfFlux"],
        np.nan,
    )

    out["x_y_err"] = np.sqrt(
        out["xErr"]**2 + out["yErr"]**2
    )

    return out


df = build_alert_features(target_alerts)

print(f"Alert-level rows: {len(df):,}")
df.head()


## 6. Build one Second-Pass row per target

In [ ]:

def calculate_stats(df, target_meta):
    grouped = df.groupby("target_id")

    summary = grouped.agg(
        # Investigation notebook definitions
        n_detections=("x", "count"),
        n_lsst_alerts=("alert_id", "size"),

        x_std=("x", "std"),
        y_std=("y", "std"),
        mean_x_y_err=("x_y_err", "mean"),
        median_x_y_err=("x_y_err", "median"),

        moment_ext_mean=("moment_ext", "mean"),
        moment_ext_std=("moment_ext", "std"),
        extendedness_mean=("extendedness", "mean"),
        extendedness_std=("extendedness", "std"),
        dipoleLength_mean=("dipoleLength", "mean"),
        dipoleLength_std=("dipoleLength", "std"),

        ra=("ra", "first"),
        dec=("dec", "first"),

        # Useful output metadata
        mjd_min=("mjd", "min"),
        mjd_max=("mjd", "max"),
    ).reset_index()

    summary["centroid_std"] = np.sqrt(
        summary["x_std"]**2
        + summary["y_std"]**2
    )

    summary["centroid_instability"] = np.where(
        summary["median_x_y_err"] > 0,
        summary["centroid_std"]
        / summary["median_x_y_err"],
        0.0,
    )

    # Same single-detection handling as investigation notebook
    summary["centroid_std"] = (
        summary["centroid_std"].fillna(0.0)
    )
    summary["centroid_instability"] = (
        summary["centroid_instability"].fillna(0.0)
    )

    for band in "ugrizy":
        band_data = (
            df[df["band"] == band]
            .groupby("target_id")
            .agg(
                **{
                    f"apFlux_mean_{band}": (
                        "apFlux", "mean"
                    ),
                    f"apFlux_std_{band}": (
                        "apFlux", "std"
                    ),
                    f"template_flux_mean_{band}": (
                        "template_flux", "mean"
                    ),
                    f"flux_ext_mean_{band}": (
                        "flux_ext", "mean"
                    ),
                    f"flux_ext_std_{band}": (
                        "flux_ext", "std"
                    ),
                }
            )
        )

        summary = summary.merge(
            band_data,
            left_on="target_id",
            right_index=True,
            how="left",
        )

    meta = target_meta[
        [
            "target_id", "ra", "dec",
            "n_loci", "member_locus_ids",
            "first_cut_max_score",
            "n_tagged",
        ]
    ].rename(
        columns={
            "ra": "target_ra",
            "dec": "target_dec",
        }
    )

    summary = summary.merge(
        meta,
        on="target_id",
        how="left",
    )

    # Use the mean locus position as the target coordinate.
    summary["ra"] = summary["target_ra"]
    summary["dec"] = summary["target_dec"]

    # Investigation notebook definition
    summary["percent_tagged"] = np.where(
        summary["n_detections"] > 0,
        np.round(
            100.0
            * summary["n_tagged"]
            / summary["n_detections"],
            5,
        ),
        np.nan,
    )

    return summary


summary = calculate_stats(df, targets)

# Diagnostic only: n_detections and n_lsst_alerts are expected to
# be the same when x is present for every retained LSST alert.
count_mismatch = (
    summary["n_detections"]
    != summary["n_lsst_alerts"]
)

print(f"Second-Pass rows: {len(summary):,}")
print(
    "n_detections != n_lsst_alerts: "
    f"{count_mismatch.sum():,}"
)

if count_mismatch.any():
    display(
        summary.loc[
            count_mismatch,
            [
                "target_id",
                "member_locus_ids",
                "n_tagged",
                "n_detections",
                "n_lsst_alerts",
                "percent_tagged",
            ],
        ]
    )

# A target with zero valid x values cannot reproduce the
# investigation notebook's percent_tagged definition.
zero_detection = summary["n_detections"] == 0

if zero_detection.any():
    display(
        summary.loc[
            zero_detection,
            [
                "target_id",
                "member_locus_ids",
                "n_tagged",
                "n_detections",
                "n_lsst_alerts",
            ],
        ]
    )

summary = summary[~zero_detection].copy()

print(f"Targets retained for scoring: {len(summary):,}")
summary.head()


## 7. Apply the Second-Pass model

In [ ]:

# Feature order produced by investigating_second_pass.ipynb
DEFAULT_MODEL_FEATURES = [
    "x_std",
    "y_std",
    "mean_x_y_err",
    "median_x_y_err",
    "moment_ext_mean",
    "moment_ext_std",
    "extendedness_mean",
    "extendedness_std",
    "dipoleLength_mean",
    "dipoleLength_std",
    "centroid_std",
    "centroid_instability",

    "apFlux_mean_u",
    "apFlux_std_u",
    "template_flux_mean_u",
    "flux_ext_mean_u",
    "flux_ext_std_u",

    "apFlux_mean_g",
    "apFlux_std_g",
    "template_flux_mean_g",
    "flux_ext_mean_g",
    "flux_ext_std_g",

    "apFlux_mean_r",
    "apFlux_std_r",
    "template_flux_mean_r",
    "flux_ext_mean_r",
    "flux_ext_std_r",

    "apFlux_mean_i",
    "apFlux_std_i",
    "template_flux_mean_i",
    "flux_ext_mean_i",
    "flux_ext_std_i",

    "apFlux_mean_z",
    "apFlux_std_z",
    "template_flux_mean_z",
    "flux_ext_mean_z",
    "flux_ext_std_z",

    "apFlux_mean_y",
    "apFlux_std_y",
    "template_flux_mean_y",
    "flux_ext_mean_y",
    "flux_ext_std_y",

    "n_tagged",
    "percent_tagged",
]


def get_model_features(model):
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    try:
        names = model.get_booster().feature_names
        if names:
            return list(names)
    except Exception:
        pass

    return DEFAULT_MODEL_FEATURES


model_features = get_model_features(model)

missing = [
    name
    for name in model_features
    if name not in summary.columns
]
if missing:
    raise KeyError(
        "Model requires features not constructed here:\n"
        + "\n".join(missing)
    )

X = summary[
    model_features
].replace(
    [np.inf, -np.inf],
    np.nan,
)

print(f"Model input features: {len(model_features)}")

summary["second_pass_score"] = (
    model.predict_proba(X)[:, 1]
)

summary["candidate_flag"] = (
    summary["second_pass_score"]
    > SECOND_PASS_THRESHOLD
)

candidates = (
    summary[summary["candidate_flag"]]
    .sort_values(
        "second_pass_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print(
    f"Candidates: {len(candidates):,} / {len(summary):,} "
    f"(score > {SECOND_PASS_THRESHOLD})"
)

print(
    "Multi-locus candidates: "
    f"{(candidates['n_loci'] > 1).sum():,}"
)


In [ ]:
candidate_columns = [
    "target_id",
    "member_locus_ids",
    "n_loci",
    "ra",
    "dec",
    "first_cut_max_score",
    "n_tagged",
    "n_detections",
    "n_lsst_alerts",
    "percent_tagged",
    "mjd_min",
    "mjd_max",
    "second_pass_score",
]

candidates[candidate_columns]

## 8. Save outputs

In [ ]:
candidate_ids = set(
    candidates["target_id"]
)

candidate_alerts = target_alerts[
    target_alerts["target_id"].isin(candidate_ids)
].copy()

candidate_membership = membership[
    membership["target_id"].isin(candidate_ids)
].copy()

summary.to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_scored_targets.parquet",
    ),
    index=False,
)

candidates[candidate_columns].to_csv(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates.csv",
    ),
    index=False,
)

candidates[candidate_columns].to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidates.parquet",
    ),
    index=False,
)

candidate_alerts.to_parquet(
    os.path.join(
        OUTPUT_DIR,
        "second_pass_candidate_alerts.parquet",
    ),
    index=False,
)

candidate_membership.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "candidate_target_membership.csv",
    ),
    index=False,
)

print("Saved:")
print("  second_pass_flagged_loci.csv")
print("  close_locus_pairs_3arcsec.csv")
print("  target_membership.csv")
print("  all_targets.parquet")
print("  second_pass_scored_targets.parquet")
print("  second_pass_candidates.csv")
print("  second_pass_candidates.parquet")
print("  second_pass_candidate_alerts.parquet")
print("  candidate_target_membership.csv")

## Optional: score distribution

In [ ]:

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

plt.figure(figsize=(4, 3))

plt.hist(
    summary["second_pass_score"],
    bins=30,
    histtype="step",
)

plt.axvline(
    SECOND_PASS_THRESHOLD,
    ls="--",
)

plt.xlabel("Second-Pass score")
plt.ylabel("Targets")
plt.tight_layout()
plt.show()


In [ ]:
multi = targets_all[targets_all["n_loci"] > 1]

print("Multi-locus targets:", len(multi))
print(multi[["target_id", "n_loci", "member_locus_ids"]])

### Notes

- The default model remains `second_pass_first_test.pkl`, as in `investigating_second_pass.ipynb`.
- The First-Cut consistency selection remains `max_score > 0.9691`.
- The May 27 alert restriction is unchanged from v7.
- Locus-level counter anomalies are excluded before target grouping, as in v7.
- **v8 target definition:** eligible loci separated by less than 3″ are linked with FoF; one connected component is one target.
- For an isolated locus, one locus is one target.
- For a multi-locus target, all alerts from all member loci are combined before feature calculation.
- `n_tagged` for a multi-locus target is the sum of the member-locus counters.
- `first_cut_max_score` is the maximum score among member loci.
- `n_detections = count(non-null x)` and `percent_tagged = 100*n_tagged/n_detections`, matching the investigation notebook.
- `n_detections != n_lsst_alerts` remains a diagnostic only.
- Sampling is performed after grouping, so `SAMPLE_FRAC=0.10` means 10% of targets.
- Candidate selection remains `second_pass_score > 0.8`.